# 📰 Agente de Notícias Policiais para WhatsApp — 100% Python, ZERO custo

Versão atualizada com 3 novas regras:
- Só considera notícias publicadas dentro de uma **janela de tempo** (padrão: últimas 48h)
- **Para automaticamente** assim que atingir a meta de notícias (padrão: 10)
- **Encurta os links** finais (API gratuita do TinyURL, sem chave)

Continua 100% Python, sem IA generativa e sem nenhuma API paga.

Rode as células em ordem, de cima pra baixo (Shift + Enter).

## Passo 1 — Instalar as dependências (gratuitas)

`feedparser` lê o RSS do Google Notícias. `python-dateutil` interpreta datas. `requests` chama o encurtador de link. Nenhuma exige chave de API.

In [11]:
!pip install feedparser python-dateutil requests googlenewsdecoder -q
print("Instalado com sucesso ✅ (sem custo, sem chave de API)")

Instalado com sucesso ✅ (sem custo, sem chave de API)


## Passo 2 — Configuração: categorias, palavras-chave, janela de tempo e meta

Este é o bloco que você vai editar com mais frequência.

- **Categorias/termos:** cada categoria vira uma seção da mensagem; cada termo é uma busca separada.
- **JANELA_HORAS_MAXIMA:** só entram notícias publicadas dentro desse número de horas (padrão: 48h).
- **META_TOTAL_NOTICIAS:** a busca para automaticamente ao atingir esse total (padrão: 10).

In [12]:
NOME_ORGANIZACAO = "Newsletter Diária - Fórum Integra"
CIDADE_TIMESTAMP = "BRASIL"

CATEGORIAS_DE_BUSCA = {
    "OPERAÇÕES POLICIAIS": [
        "operação policial prisão",
        "operação policial tráfico de drogas",
        "operação policial facção criminosa",
    ],
    "CRIMES E INVESTIGAÇÕES": [
        "polícia civil investigação homicídio",
        "polícia prende suspeito",
        "polícia investiga fraude",
    ],
    "APREENSÕES": [
        "polícia apreensão armas",
        "polícia apreensão drogas",
    ],
    "SEGURANÇA PÚBLICA - GERAL": [
        "segurança pública Brasil",
        "polícia militar ação",
        "polícia civil ação",
    ],
}

# Parâmetros do pipeline
MAX_RESULTADOS_POR_TERMO = 15
JANELA_HORAS_MAXIMA = 24          # só notícias das últimas 24h
META_TOTAL_NOTICIAS = 10           # para a busca ao atingir este total
LIMIAR_SIMILARIDADE_TITULO = 0.62
TOLERANCIA_DIAS_ENTRE_DUPLICATAS = 1
ATIVAR_ENCURTADOR_DE_LINK = False  # False = mantem o link original (recomendado contra phishing)
TAMANHO_MAXIMO_POR_PARTE = 3500

print("Configuração carregada ✅")
print(f"Categorias: {list(CATEGORIAS_DE_BUSCA.keys())}")
print(f"Janela de tempo: últimas {JANELA_HORAS_MAXIMA}h | Meta: {META_TOTAL_NOTICIAS} notícia(s)")

Configuração carregada ✅
Categorias: ['OPERAÇÕES POLICIAIS', 'CRIMES E INVESTIGAÇÕES', 'APREENSÕES', 'SEGURANÇA PÚBLICA - GERAL']
Janela de tempo: últimas 48h | Meta: 10 notícia(s)


## 🧩 Agente — Deduplicador

Precisa ser carregado ANTES do buscador, porque a busca já filtra duplicatas na hora (parada antecipada). Decide se duas notícias falam do mesmo fato cruzando data, localidade e semelhança do título. **Sem IA — só `difflib`, biblioteca padrão do Python.**

In [13]:
import re
from difflib import SequenceMatcher
from dateutil import parser as date_parser

ESTADOS_BRASILEIROS = {
    "acre": "AC", "alagoas": "AL", "amapá": "AP", "amazonas": "AM", "bahia": "BA",
    "ceará": "CE", "distrito federal": "DF", "espírito santo": "ES", "goiás": "GO",
    "maranhão": "MA", "mato grosso do sul": "MS", "mato grosso": "MT",
    "minas gerais": "MG", "pará": "PA", "paraíba": "PB", "paraná": "PR",
    "pernambuco": "PE", "piauí": "PI", "rio de janeiro": "RJ",
    "rio grande do norte": "RN", "rio grande do sul": "RS", "rondônia": "RO",
    "roraima": "RR", "santa catarina": "SC", "são paulo": "SP", "sergipe": "SE",
    "tocantins": "TO",
}

PALAVRAS_IGNORADAS = {
    "a", "o", "as", "os", "de", "da", "do", "das", "dos", "em", "no", "na",
    "nos", "nas", "e", "é", "um", "uma", "para", "por", "com", "que", "se",
    "sobre", "após", "contra",
}


def extrair_localidade(texto: str) -> str:
    texto_lower = texto.lower()
    for nome_estado, sigla in ESTADOS_BRASILEIROS.items():
        if nome_estado in texto_lower:
            return sigla
    correspondencia = re.search(r"\b([A-Z]{2})\b", texto)
    if correspondencia and correspondencia.group(1) in ESTADOS_BRASILEIROS.values():
        return correspondencia.group(1)
    return "não identificada"


def normalizar_data(data_bruta: str):
    if not data_bruta:
        return None
    try:
        return date_parser.parse(data_bruta, fuzzy=True).date()
    except (ValueError, OverflowError):
        return None


def calcular_similaridade(titulo_a: str, titulo_b: str) -> float:
    return SequenceMatcher(None, titulo_a.lower(), titulo_b.lower()).ratio()


def calcular_sobreposicao_palavras(titulo_a: str, titulo_b: str) -> float:
    def palavras_significativas(texto):
        brutas = re.findall(r"[a-zà-ú0-9]+", texto.lower())
        return {p for p in brutas if p not in PALAVRAS_IGNORADAS and len(p) > 2}
    conjunto_a = palavras_significativas(titulo_a)
    conjunto_b = palavras_significativas(titulo_b)
    if not conjunto_a or not conjunto_b:
        return 0.0
    return len(conjunto_a & conjunto_b) / len(conjunto_a | conjunto_b)


def sao_a_mesma_noticia(noticia_a, noticia_b, limiar_similaridade=0.62, tolerancia_dias=1) -> bool:
    data_a = normalizar_data(noticia_a.get("data_publicacao", ""))
    data_b = normalizar_data(noticia_b.get("data_publicacao", ""))
    if data_a and data_b and abs((data_a - data_b).days) > tolerancia_dias:
        return False

    localidade_a = extrair_localidade(noticia_a["titulo"])
    localidade_b = extrair_localidade(noticia_b["titulo"])
    if localidade_a != "não identificada" and localidade_b != "não identificada" and localidade_a != localidade_b:
        return False

    sim_caracteres = calcular_similaridade(noticia_a["titulo"], noticia_b["titulo"])
    sobreposicao = calcular_sobreposicao_palavras(noticia_a["titulo"], noticia_b["titulo"])
    return sim_caracteres >= limiar_similaridade or sobreposicao >= 0.4


print("Agente Deduplicador carregado ✅")

Agente Deduplicador carregado ✅


## 🕵️ Agente 1 — Buscador (com janela de tempo e parada automática)

Consulta o RSS do Google Notícias termo por termo. Para CADA notícia encontrada, verifica na hora:
1. Está dentro da janela de tempo configurada? (senão, descarta)
2. É duplicata de alguma já coletada? (senão, descarta)

Assim que atingir a meta, **para imediatamente** — não gasta tempo com os termos restantes.

In [14]:
import time
import urllib.parse
from datetime import datetime, timedelta, timezone
import feedparser

CABECALHOS = {
    "User-Agent": "Mozilla/5.0 (compatible; AgenteNoticiasINTEGRA/1.0; +https://forumintegra.org)"
}
PAUSA_ENTRE_BUSCAS_SEGUNDOS = 1.5


def dentro_da_janela_de_tempo(data_publicacao: str, horas_maximas: int = 48) -> bool:
    if not data_publicacao:
        return False
    try:
        data = date_parser.parse(data_publicacao, fuzzy=True)
    except (ValueError, OverflowError):
        return False
    if data.tzinfo is None:
        data = data.replace(tzinfo=timezone.utc)
    agora = datetime.now(timezone.utc)
    diferenca = agora - data
    return timedelta(0) <= diferenca <= timedelta(hours=horas_maximas)


def buscar_por_termo(termo: str, max_resultados: int = 15) -> list:
    consulta = urllib.parse.quote(termo)
    url = f"https://news.google.com/rss/search?q={consulta}&hl=pt-BR&gl=BR&ceid=BR:pt-419"
    feed = feedparser.parse(url, request_headers=CABECALHOS)

    resultados = []
    for entrada in feed.entries[:max_resultados]:
        fonte = "Fonte não identificada"
        if hasattr(entrada, "source") and hasattr(entrada.source, "title"):
            fonte = entrada.source.title
        resultados.append({
            "titulo": entrada.get("title", "").strip(),
            "link": entrada.get("link", "").strip(),
            "fonte": fonte,
            "data_publicacao": entrada.get("published", ""),
            "termo_busca": termo,
        })
    return resultados


def buscar_ate_atingir_meta(categorias: dict, meta_total: int = 10, janela_horas: int = 48, max_por_termo: int = 15) -> list:
    coletadas = []
    for nome_categoria, termos in categorias.items():
        print(f"[Buscador] Categoria: {nome_categoria}")
        for termo in termos:
            print(f"   buscando: '{termo}'...")
            resultados = buscar_por_termo(termo, max_por_termo)

            for noticia in resultados:
                if len(coletadas) >= meta_total:
                    break
                if not dentro_da_janela_de_tempo(noticia["data_publicacao"], janela_horas):
                    continue
                noticia["categoria"] = nome_categoria
                eh_duplicada = any(sao_a_mesma_noticia(noticia, existente) for existente in coletadas)
                if not eh_duplicada:
                    coletadas.append(noticia)

            if len(coletadas) >= meta_total:
                print(f"[Buscador] Meta de {meta_total} notícia(s) atingida. Parando busca.")
                return coletadas

            time.sleep(PAUSA_ENTRE_BUSCAS_SEGUNDOS)

    if len(coletadas) < meta_total:
        print(f"[Buscador] Aviso: esgotou os termos e encontrou apenas {len(coletadas)} de {meta_total} "
              f"notícia(s) dentro de {janela_horas}h. Considere adicionar mais termos ou aumentar a janela.")
    return coletadas


print("Agente 1 (Buscador) carregado ✅")

Agente 1 (Buscador) carregado ✅


## 🔗 Agente — Resolvedor de link original

O RSS do Google Notícias entrega links "embrulhados" (`news.google.com/rss/articles/...`) que escondem o domínio de destino até o clique — o mesmo problema de segurança que motivou desativar o encurtador. Este agente decodifica esses links usando a biblioteca gratuita `googlenewsdecoder`, recuperando o link real da matéria (ex: `g1.globo.com/...`).

In [ ]:
from googlenewsdecoder import gnewsdecoder

INTERVALO_ENTRE_DECODIFICACOES_SEGUNDOS = 1


def resolver_link_original(url: str) -> str:
    if not url or "news.google.com" not in url:
        return url
    try:
        resultado = gnewsdecoder(url, interval=INTERVALO_ENTRE_DECODIFICACOES_SEGUNDOS)
        if resultado.get("status") and resultado.get("decoded_url"):
            return resultado["decoded_url"]
    except Exception:
        pass
    return url


def resolver_links_das_noticias(noticias: list) -> list:
    for noticia in noticias:
        noticia["link"] = resolver_link_original(noticia["link"])
    return noticias


print("Agente Resolvedor de Link carregado ✅")

## ✂️ Agente 2 — Encurtador de links

Usa a API pública e gratuita do TinyURL para encurtar os links das notícias já selecionadas. Se o serviço falhar por qualquer motivo, o link original é mantido — o pipeline nunca quebra por causa disso.

In [15]:
import requests

URL_API_TINYURL = "https://tinyurl.com/api-create.php"
TIMEOUT_SEGUNDOS = 5


def encurtar_link(url_original: str) -> str:
    if not url_original:
        return url_original
    try:
        resposta = requests.get(URL_API_TINYURL, params={"url": url_original}, timeout=TIMEOUT_SEGUNDOS)
        texto = resposta.text.strip()
        if resposta.status_code == 200 and texto.startswith("http"):
            return texto
    except requests.RequestException:
        pass
    return url_original


def encurtar_links_das_noticias(noticias: list, ativar_encurtador: bool = False) -> list:
    # Por padrao, mantem o link original -- encurtadores escondem o
    # dominio de destino, o que pode ser lido como phishing por um
    # publico que precisa inspecionar links antes de clicar.
    for noticia in noticias:
        if ativar_encurtador:
            noticia["link_curto"] = encurtar_link(noticia["link"])
        else:
            noticia["link_curto"] = noticia["link"]
    return noticias


print("Agente 2 (Encurtador) carregado ✅")

Agente 2 (Encurtador) carregado ✅


## 🗂️ Agente 3 — Organizador

Agrupa as notícias selecionadas nas categorias definidas no Passo 2, mantendo a ordem configurada.

In [16]:
def organizar_por_categoria(noticias: list, categorias_config: dict) -> dict:
    organizado = {nome: [] for nome in categorias_config}
    for noticia in noticias:
        categoria = noticia.get("categoria", "OUTROS")
        organizado.setdefault(categoria, [])
        organizado[categoria].append(noticia)
    return {nome: lista for nome, lista in organizado.items() if lista}


print("Agente 3 (Organizador) carregado ✅")

Agente 3 (Organizador) carregado ✅


## 🎨 Agente 4 — Formatador de mensagem WhatsApp

Monta o texto final no formato do seu modelo, usando o **link curto** (com o link original como reserva, caso o encurtamento tenha falhado).

In [17]:
from datetime import datetime as dt

MESES_EM_PORTUGUES = {
    1: "janeiro", 2: "fevereiro", 3: "março", 4: "abril", 5: "maio", 6: "junho",
    7: "julho", 8: "agosto", 9: "setembro", 10: "outubro", 11: "novembro", 12: "dezembro",
}


def _montar_cabecalho(nome_organizacao, cidade, numero_parte, total_partes):
    agora = dt.now()
    linhas = [
        f"🖥 - *{nome_organizacao}*",
        f"*{cidade},* *{agora.day}* *de* *{MESES_EM_PORTUGUES[agora.month]}* *de* *{agora.year}*",
    ]
    if total_partes > 1:
        linhas.append(f"*Parte {numero_parte:02d}*")
    linhas.append("")
    return "\n".join(linhas)


def _montar_secao(nome_categoria, noticias):
    linhas = [f" *===* *{nome_categoria}* *===*", "====================================="]
    for noticia in noticias:
        linhas.append(f"*{noticia['titulo']}*")
        linhas.append(noticia.get("link_curto") or noticia["link"])
        linhas.append("--------------------------------------")
    linhas.append("")
    return "\n".join(linhas)


def gerar_mensagem_completa(noticias_organizadas, nome_organizacao, cidade, tamanho_maximo_por_parte=3500):
    blocos = [_montar_secao(cat, noticias) for cat, noticias in noticias_organizadas.items()]

    partes_de_conteudo = []
    parte_atual = ""
    for bloco in blocos:
        if parte_atual and len(parte_atual) + len(bloco) > tamanho_maximo_por_parte:
            partes_de_conteudo.append(parte_atual)
            parte_atual = bloco
        else:
            parte_atual += bloco
    if parte_atual:
        partes_de_conteudo.append(parte_atual)
    if not partes_de_conteudo:
        partes_de_conteudo = ["Nenhuma notícia relevante encontrada nesta busca."]

    total_partes = len(partes_de_conteudo)
    mensagens_finais = []
    for indice, conteudo in enumerate(partes_de_conteudo, start=1):
        cabecalho = _montar_cabecalho(nome_organizacao, cidade, indice, total_partes)
        mensagens_finais.append(f"{cabecalho}\n{conteudo}")
    return mensagens_finais


print("Agente 4 (Formatador) carregado ✅")

Agente 4 (Formatador) carregado ✅


## 🚀 Rodando o pipeline completo

Busca (com janela de tempo + parada na meta) → encurta links → organiza → formata.

In [18]:
def executar_pipeline():
    print(f"[1/4] Buscando notícias das últimas {JANELA_HORAS_MAXIMA}h (meta: {META_TOTAL_NOTICIAS})...")
    noticias_selecionadas = buscar_ate_atingir_meta(
        CATEGORIAS_DE_BUSCA,
        meta_total=META_TOTAL_NOTICIAS,
        janela_horas=JANELA_HORAS_MAXIMA,
        max_por_termo=MAX_RESULTADOS_POR_TERMO,
    )
    print(f"       -> {len(noticias_selecionadas)} notícia(s) selecionada(s).")

    print("[2/5] Resolvendo o link real das matérias (removendo redirecionamento do Google Notícias)...")
    noticias_selecionadas = resolver_links_das_noticias(noticias_selecionadas)

    print(f"[3/5] Processando links (encurtador {'ativado' if ATIVAR_ENCURTADOR_DE_LINK else 'desativado -- usando link original'})...")
    noticias_selecionadas = encurtar_links_das_noticias(noticias_selecionadas, ativar_encurtador=ATIVAR_ENCURTADOR_DE_LINK)

    print("[3/4] Organizando por categoria...")
    noticias_organizadas = organizar_por_categoria(noticias_selecionadas, CATEGORIAS_DE_BUSCA)

    print("[4/4] Gerando mensagem(ns) final(is) para WhatsApp...")
    mensagens = gerar_mensagem_completa(
        noticias_organizadas,
        nome_organizacao=NOME_ORGANIZACAO,
        cidade=CIDADE_TIMESTAMP,
        tamanho_maximo_por_parte=TAMANHO_MAXIMO_POR_PARTE,
    )
    return mensagens


mensagens_finais = executar_pipeline()

print("\n" + "=" * 60)
print("MENSAGEM(NS) PRONTA(S):")
print("=" * 60 + "\n")
for mensagem in mensagens_finais:
    print(mensagem)
    print("\n" + "-" * 60 + "\n")

[1/4] Buscando notícias das últimas 48h (meta: 10)...
[Buscador] Categoria: OPERAÇÕES POLICIAIS
   buscando: 'operação policial prisão'...
   buscando: 'operação policial tráfico de drogas'...
[Buscador] Meta de 10 notícia(s) atingida. Parando busca.
       -> 10 notícia(s) selecionada(s).
[2/4] Encurtando links...
[3/4] Organizando por categoria...
[4/4] Gerando mensagem(ns) final(is) para WhatsApp...

MENSAGEM(NS) PRONTA(S):

🖥 - *NEWSLETTER DIÁRIA FÓRUM INTEGRA*
*BRASIL,* *28* *July*, *2026*  *01:20*

 *===* *OPERAÇÕES POLICIAIS* *===*
*Bandidos fuzilam base da polícia na Gardênia Azul após prisão de traficante; PM cita ‘sanha expansionista’ do crime - G1*
https://tinyurl.com/2cpdfkqs
--------------------------------------
*Três advogados e um PM integram grupo de agiotagem e extorsão - amazonas atual*
https://tinyurl.com/259fqjej
--------------------------------------
*Suspeito é preso por tráfico de drogas durante operação policial na Praia de Mangabeira, em Ponta de Pedras - notí

## 💾 Baixar a mensagem final

Salva o texto e baixa direto pro seu computador — é só abrir o `.txt`, copiar e colar no WhatsApp.

In [19]:
from google.colab import files

nome_arquivo = "mensagem_whatsapp.txt"
with open(nome_arquivo, "w", encoding="utf-8") as f:
    f.write("\n\n".join(mensagens_finais))

files.download(nome_arquivo)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## ⚠️ Notas importantes

- **Janela de tempo:** notícias sem data reconhecível são descartadas por segurança (preferimos perder uma notícia duvidosa a incluir algo desatualizado).
- **Meta de notícias:** se a busca esgotar todos os termos configurados e não atingir a meta, você verá um aviso no console — nesse caso, adicione mais termos no Passo 2 ou aumente `JANELA_HORAS_MAXIMA`.
- **Encurtador de link:** se o TinyURL estiver temporariamente fora do ar, o link original aparece na mensagem — o pipeline não quebra.
- **Filtro de duplicatas:** continua sendo uma abordagem determinística (sem IA). Pode, em casos raros, deixar passar duas notícias do mesmo fato com vocabulário muito diferente entre si.